# Задача 3. AVL

Реализовать класс `AVL`, который представляет собой AVL-дерево. Поддержать операции вставки, удаления и поиска. Написать тесты, включая граничные случаи, визуализацию деревьев на примерах и оценку сложности.

## Идея

AVL — это BST, у которого для каждого узла высоты левого и правого поддеревьев отличаются не более чем на 1. В каждом узле храним его высоту. После вставки или удаления возвращаемся по рекурсивным вызовам к корню, обновляем высоты и при необходимости выполняем поворот. Поворот меняет связи между несколькими узлами, но сохраняет порядок BST.

Баланс узла — `высота слева − высота справа`. Если он больше `1`, дерево перевешивает влево; если меньше `−1`, вправо. Для случаев «лево-право» и «право-лево» делаем два поворота. Повторные значения не вставляем, поиск возвращает `True` или `False`.

## Реализация

In [1]:
class AVLNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None
        self.height = 1


class AVL:
    def __init__(self):
        self.root = None

    def _height(self, node):
        return node.height if node is not None else 0

    def _update_height(self, node):
        node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _balance(self, node):
        return self._height(node.left) - self._height(node.right)

    def _rotate_right(self, node):
        new_root = node.left
        middle = new_root.right
        new_root.right = node
        node.left = middle
        self._update_height(node)
        self._update_height(new_root)
        return new_root

    def _rotate_left(self, node):
        new_root = node.right
        middle = new_root.left
        new_root.left = node
        node.right = middle
        self._update_height(node)
        self._update_height(new_root)
        return new_root

    def _rebalance(self, node):
        self._update_height(node)
        balance = self._balance(node)

        if balance > 1:
            if self._balance(node.left) < 0:
                node.left = self._rotate_left(node.left)
            return self._rotate_right(node)

        if balance < -1:
            if self._balance(node.right) > 0:
                node.right = self._rotate_right(node.right)
            return self._rotate_left(node)

        return node

    def _insert(self, node, value):
        if node is None:
            return AVLNode(value)

        if value < node.value:
            node.left = self._insert(node.left, value)
        elif value > node.value:
            node.right = self._insert(node.right, value)
        else:
            return node

        return self._rebalance(node)

    def insert(self, value):
        self.root = self._insert(self.root, value)

    def search(self, value):
        current = self.root
        while current is not None:
            if value == current.value:
                return True
            current = current.left if value < current.value else current.right
        return False

    def _smallest(self, node):
        while node.left is not None:
            node = node.left
        return node

    def _delete(self, node, value):
        if node is None:
            return None

        if value < node.value:
            node.left = self._delete(node.left, value)
        elif value > node.value:
            node.right = self._delete(node.right, value)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left

            successor = self._smallest(node.right)
            node.value = successor.value
            node.right = self._delete(node.right, successor.value)

        return self._rebalance(node)

    def delete(self, value):
        self.root = self._delete(self.root, value)

## Пример работы

Вставим `30`, `10`, `20`. После обычной BST-вставки получается перекос «лево-право»:

```text
    30
   /
  10
    \
     20
```

Сначала поворачиваем поддерево с корнем `10` влево, затем дерево с корнем `30` вправо:

```text
    20
   /  \
  10  30
```

При удалении тоже проверяем баланс на обратном пути. Например, после удаления `40` из дерева слева корень `30` перевесит влево; правый поворот даст дерево справа:

```text
    30               20
   /  \             /  \
  20  40     →      10   30
 /  \                   /
10  25                 25
```

## Сложность

Высота AVL-дерева равна `O(log n)`, где `n` — число узлов. Поиск, вставка и удаление проходят один путь от корня и занимают `O(log n)` времени; на обратном пути в каждом узле выполняется не более пары поворотов за `O(1)`. Поиск реализован циклом и использует `O(1)` дополнительной памяти. Вставка и удаление используют `O(log n)` памяти на стек рекурсии. Само дерево хранит `O(n)` узлов.

## Тесты

Проверяем пустое дерево, поиск, дубликат, все четыре вида поворотов при вставке, удаление листа, узла с одним и двумя детьми, удаление корня и отсутствующего значения. После каждого действия проверяем не только найденные значения, но и высоты, порядок BST и баланс каждого узла.

In [2]:
def check_avl(node, lower=None, upper=None):
    if node is None:
        return 0
    if lower is not None:
        assert node.value > lower
    if upper is not None:
        assert node.value < upper

    left_height = check_avl(node.left, lower, node.value)
    right_height = check_avl(node.right, node.value, upper)
    assert abs(left_height - right_height) <= 1
    assert node.height == 1 + max(left_height, right_height)
    return node.height


empty = AVL()
assert empty.search(1) is False
empty.delete(1)
assert empty.root is None
empty.insert(1)
assert empty.search(1) is True
empty.insert(1)
assert empty.root.left is None and empty.root.right is None
check_avl(empty.root)

for values in ((30, 20, 10), (10, 20, 30),
               (30, 10, 20), (10, 30, 20)):
    tree = AVL()
    for value in values:
        tree.insert(value)
        check_avl(tree.root)
    assert tree.root.value == 20
    assert tree.root.left.value == 10
    assert tree.root.right.value == 30

one_child = AVL()
one_child.insert(2)
one_child.insert(1)
one_child.delete(2)
assert one_child.root.value == 1
check_avl(one_child.root)

tree = AVL()
for value in (20, 10, 30, 5, 15, 25, 40):
    tree.insert(value)
tree.delete(20)  # корень с двумя детьми
assert tree.search(20) is False
assert tree.search(25) is True
check_avl(tree.root)
tree.delete(999)  # отсутствующее значение
check_avl(tree.root)

tree = AVL()
for value in range(1, 21):
    tree.insert(value)
    check_avl(tree.root)
for value in range(1, 21):
    assert tree.search(value) is True
    tree.delete(value)
    assert tree.search(value) is False
    check_avl(tree.root)
assert tree.root is None

print("Задача 3: все тесты пройдены")

Задача 3: все тесты пройдены
